# Continue CookLens training with the data add-on

1. Keep **cooklens-67-inputs.zip** in the top level of **My Drive**.
2. After reviewing the new photos, upload **cooklens-67-round2-addon.zip** to the same location. Leave both ZIPs intact.
3. Set `source_run` in the first code cell to the epoch 4 run folder in `My Drive/CookLens-67/runs`.
4. Choose **Runtime → Change runtime type → GPU**, then **Runtime → Run all**.

The add-on supplies only the additional photos and their split assignments. Colab reuses the original archive and loads your epoch 4 weights directly from the previous run in Drive. The selected checkpoint and report go into a new folder under **My Drive/CookLens-67/runs**.

## 1. Check CUDA and load the data bundle

In [ ]:
from contextlib import ExitStack
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import shutil
from tempfile import TemporaryDirectory
from time import perf_counter
from zipfile import ZipFile

import torch
import torchvision
from google.colab import drive

class Progress:
    def __init__(self, label, total, unit="photos", scale=1):
        self.label, self.total, self.unit, self.scale = label, total, unit, scale
        self.start = self.last = perf_counter()
        print(f"{label} | starting {total / scale:,.1f} {unit}" if scale != 1
              else f"{label} | starting {total:,} {unit}", flush=True)

    def due(self, completed):
        return completed >= self.total or perf_counter() - self.last >= 5

    def update(self, completed, detail=""):
        now = perf_counter()
        if completed < self.total and now - self.last < 5:
            return
        elapsed = max(now - self.start, 0.001)
        rate = completed / elapsed
        eta = int(max(0, self.total - completed) / max(rate, 0.001))
        if self.scale == 1:
            amount = f"{completed:,}/{self.total:,}"
        else:
            amount = f"{completed / self.scale:,.1f}/{self.total / self.scale:,.1f}"
        time_status = f"{elapsed:.1f}s elapsed" if completed == self.total else f"ETA {eta // 60}m {eta % 60:02d}s"
        percent = 100 * completed // self.total
        print(f"{self.label} | {amount} {self.unit} | {percent}% | "
              f"{rate / self.scale:,.1f} {self.unit}/s | {time_status}{detail}", flush=True)
        self.last = now


if not torch.cuda.is_available():
    raise RuntimeError("Choose Runtime > Change runtime type > GPU, then run all cells again.")
gpu = torch.cuda.get_device_properties(0)
device_name = gpu.name
gpu_gib = gpu.total_memory / 2**30
test_tensor = torch.ones((2, 2), device="cuda")
assert (test_tensor @ test_tensor).sum().item() == 8
del test_tensor
print(f"GPU ready | {device_name} | {gpu_gib:.1f} GiB | PyTorch {torch.__version__}", flush=True)

mount = Path("/content/drive")
print("Connecting Google Drive", flush=True)
drive.mount(str(mount))
my_drive = mount / "MyDrive"
output_root = my_drive / "CookLens-67" / "runs"

# Set this to the run whose selected checkpoint you want to continue.
source_run = ""
if not source_run.strip():
    raise ValueError("Set source_run to the epoch 4 run folder in My Drive/CookLens-67/runs.")
checkpoint = output_root / source_run / "efficientnet_b2_pilot.pt"
if not checkpoint.is_file():
    raise FileNotFoundError(f"Starting checkpoint is missing: {checkpoint}")

bundle = my_drive / "cooklens-67-inputs.zip"
addon_bundle = my_drive / "cooklens-67-round2-addon.zip"
if not addon_bundle.is_file():
    raise FileNotFoundError(f"Upload {addon_bundle.name} to My Drive next to {bundle.name}.")
# Optional: paste the base ZIP checksum from your private export record.
base_zip_sha256 = ""
if not bundle.is_file():
    raise FileNotFoundError(f"Upload {bundle.name} to the top level of My Drive and rerun this cell.")
bundle_info = bundle.stat()
bundle_size = bundle_info.st_size
with ZipFile(addon_bundle) as package:
    addon_manifest = json.loads(package.read("addon.json"))
base_plan_digest = addon_manifest.get("base_prepared_sha256", "")
with ZipFile(bundle) as package:
    prepared_digest = hashlib.sha256(package.read("prepared.json")).hexdigest()
    unpacked_size = sum(entry.file_size for entry in package.infolist())
if prepared_digest != base_plan_digest:
    raise ValueError("The data add-on does not match the prepared manifest in the base ZIP.")

workspace = Path("/content/cooklens-67")
workspace.mkdir(exist_ok=True)
cache_key = f"{prepared_digest}-{bundle_size}-{bundle_info.st_mtime_ns}"
if base_zip_sha256:
    cache_key += "-" + hashlib.sha256(base_zip_sha256.encode()).hexdigest()
data_dir = workspace / cache_key
if not data_dir.is_dir():
    free_disk = shutil.disk_usage(workspace).free
    if free_disk < unpacked_size + 100_000_000:
        raise OSError("The Colab VM does not have enough free space to extract the dataset.")
    with TemporaryDirectory(dir=workspace) as scratch_dir:
        scratch = Path(scratch_dir)
        copy_first = free_disk >= unpacked_size + bundle_size + 100_000_000
        archive_path = scratch / bundle.name if copy_first else bundle
        if copy_first or base_zip_sha256:
            label = "Copy base ZIP" if copy_first else "Verify base ZIP"
            read_progress = Progress(label, bundle_size, "MB", 1_000_000)
            digest = hashlib.sha256()
            transferred = 0
            with ExitStack() as streams:
                source = streams.enter_context(bundle.open("rb"))
                staged = streams.enter_context(archive_path.open("wb")) if copy_first else None
                while chunk := source.read(8 * 1024**2):
                    if base_zip_sha256:
                        digest.update(chunk)
                    if staged is not None:
                        staged.write(chunk)
                    transferred += len(chunk)
                    read_progress.update(transferred)
            if base_zip_sha256 and digest.hexdigest() != base_zip_sha256:
                raise ValueError(f"Checksum mismatch for {bundle.name}. Re-upload it from the original export.")

        unpacked = scratch / "inputs"
        with ZipFile(archive_path) as package:
            entries = package.infolist()
            extraction = Progress("Extract audited inputs", sum(entry.file_size for entry in entries),
                                  "MB", 1_000_000)
            extracted = 0
            for entry in entries:
                target = (unpacked / entry.filename).resolve()
                if not target.is_relative_to(unpacked.resolve()):
                    raise ValueError(f"Invalid path in {bundle.name}: {entry.filename}")
                if entry.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                with package.open(entry) as member, target.open("wb") as local:
                    while chunk := member.read(8 * 1024**2):
                        local.write(chunk)
                        extracted += len(chunk)
                        extraction.update(extracted)
        unpacked.rename(data_dir)
else:
    print("Reusing the audited dataset already on this Colab VM.", flush=True)

archives = data_dir / "archives"
fallback = data_dir / "fallback"
prepared = data_dir / "prepared.json"
extra_epochs = 5
feature_lr = 1e-5
head_lr = 1e-4
lr_patience = 1
stop_patience = 3
min_f1_gain = 0.001
# Reduce either batch size here if your assigned GPU runs out of memory.
batch_size = 64 if gpu_gib >= 12 else 32 if gpu_gib >= 8 else 16
eval_batch_size = batch_size * 2
cpu_count = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 1)
threads = max(1, min(4, cpu_count))
max_workers = min(4, cpu_count)
run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
output = output_root / run_id


## 2. Load the audited splits and sample training photos

In [ ]:
from collections import defaultdict
import random

from PIL import Image, ImageOps
from torch import nn
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from torchvision import transforms
from torchvision.models import EfficientNet_B2_Weights, efficientnet_b2

SPLITS = ("train", "validation", "test")
FALLBACK_ORDER = ("other_food", "non_food")
BASELINE_CLASSES = ['Banh beo',
 'Banh bot loc',
 'Banh can',
 'Banh canh',
 'Banh chung',
 'Banh cuon',
 'Banh duc',
 'Banh gio',
 'Banh khot',
 'Banh mi',
 'Banh pia',
 'Banh tet',
 'Banh trang nuong',
 'Banh xeo',
 'Bun bo Hue',
 'Bun dau mam tom',
 'Bun mam',
 'Bun rieu',
 'Bun thit nuong',
 'Ca kho to',
 'Canh chua',
 'Cao lau',
 'Chao long',
 'Com tam',
 'Goi cuon',
 'Hu tieu',
 'Mi quang',
 'Nem chua',
 'Pho',
 'Xoi xeo']
WEIGHTS = EfficientNet_B2_Weights.IMAGENET1K_V1


def file_under(root, name):
    path = (root / name).resolve()
    if not path.is_relative_to(root.resolve()) or not path.is_file():
        raise ValueError(f"Missing or invalid image path: {name}")
    return path


class PilotImages(Dataset):
    def __init__(self, rows, archive_path, fallback, addon_images, transform):
        self.rows = rows
        self.archive_path = archive_path
        self.transform = transform
        roots = {"fallback": fallback, "addon": addon_images}
        self.fallback_paths = {
            row["path"]: file_under(roots[row["source"]], row["path"])
            for row in rows if row["source"] != "baseline"
        }
        self.archive = None

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        if row["source"] == "baseline":
            # Each DataLoader worker opens its own handle to the local ZIP.
            if self.archive is None:
                self.archive = ZipFile(self.archive_path)
            stream = self.archive.open(row["path"])
        else:
            stream = self.fallback_paths[row["path"]].open("rb")
        with stream, Image.open(stream) as image:
            pixels = self.transform(ImageOps.exif_transpose(image).convert("RGB"))
        return pixels, row["class_index"]

    def close(self):
        if self.archive is not None:
            self.archive.close()
            self.archive = None


class PhaseBatches:
    """Feed the current split to the same loader between fully consumed passes."""

    def __init__(self):
        self.indices = ()
        self.batch_size = 1

    def select(self, indices, batch_size):
        self.indices = indices
        self.batch_size = batch_size

    def __iter__(self):
        size = self.batch_size
        for start in range(0, len(self.indices), size):
            yield self.indices[start:start + size]

    def __len__(self):
        return (len(self.indices) + self.batch_size - 1) // self.batch_size


def sample_epoch(groups, non_food_index, draws_per_class, seed):
    rng = random.Random(seed)
    draws = []
    for class_index, indices in sorted(groups.items()):
        remaining = draws_per_class * (2 if class_index == non_food_index else 1)
        if not indices or remaining <= 0:
            raise ValueError("Each training class must have images and a positive draw count.")
        while remaining:
            size = min(remaining, len(indices))
            draws.extend(rng.sample(indices, size))
            remaining -= size
    rng.shuffle(draws)
    return draws


def collect_rows(plan):
    baseline = plan["baseline_classes"]
    expected = set(range(len(plan["classes"])))
    result = {}
    for split in SPLITS:
        rows = [row for row in plan["images"] if row["split"] == split]
        counts = defaultdict(int)
        for row in rows:
            counts[row["class_index"]] += 1
        if set(counts) != expected or counts[baseline + 1] != 2 * counts[baseline]:
            raise ValueError(f"Class coverage or the fallback ratio is invalid in {split}.")
        result[split] = rows
    return result


def load_data_addon(prepared_path, bundle_path, workspace):
    base_bytes = prepared_path.read_bytes()
    base_plan = json.loads(base_bytes)
    base_digest = hashlib.sha256(base_bytes).hexdigest()
    counters = {(split, label): 0 for split in SPLITS for label in FALLBACK_ORDER}
    label_index = {label: base_plan["classes"].index(label) for label in FALLBACK_ORDER}
    existing_ids = {row["id"] for row in base_plan["images"] if "id" in row}
    existing_paths = {row["path"] for row in base_plan["images"]}
    group_sets = {}
    additions = []
    with bundle_path.open("rb") as source:
        digest = hashlib.file_digest(source, "sha256").hexdigest()
        source.seek(0)
        with ZipFile(source) as bundle:
            members = bundle.namelist()
            if len(set(members)) != len(members):
                raise ValueError("Duplicate archive entries in the data add-on.")
            if "addon.json" not in members:
                raise ValueError("The data add-on is missing addon.json.")
            manifest = json.loads(bundle.read("addon.json"))
            if manifest.get("schema_version") != 1 or manifest.get("base_prepared_sha256") != base_digest:
                raise ValueError("The data add-on belongs to a different base training ZIP.")
            photos = manifest.get("photos")
            if not isinstance(photos, list) or not photos:
                raise ValueError("The data add-on must list at least one photo.")

            paths = set()
            for photo in photos:
                identifier, name = photo["id"], photo["path"]
                label, split, group = photo["collection_label"], photo["split"], photo["split_group"]
                if not identifier or not identifier.isascii() or not identifier.replace("_", "").isalnum():
                    raise ValueError("Invalid add-on photo ID.")
                if name != f"images/{identifier}.jpg" or split not in SPLITS or label not in FALLBACK_ORDER or not group:
                    raise ValueError(f"Invalid add-on photo entry {identifier}.")
                if identifier in existing_ids or name in existing_paths:
                    raise ValueError(f"Duplicate add-on photo {identifier}.")
                if group_sets.setdefault(group, split) != split:
                    raise ValueError(f"Add-on photo group crosses splits: {group}.")
                existing_ids.add(identifier)
                existing_paths.add(name)
                paths.add(name)
                counters[split, label] += 1
                additions.append({"source": "addon", "path": name, "id": identifier, "split": split,
                                  "class_index": label_index[label], "origin": photo["origin"]})

            if set(members) != paths | {"addon.json"}:
                raise ValueError("The add-on file list differs from its manifest.")
            plan = {**base_plan, "images": base_plan["images"] + additions}
            split_rows = collect_rows(plan)
            plan["fallback_counts"] = {
                split: {label: sum(row["class_index"] == index for row in rows)
                        for label, index in label_index.items()}
                for split, rows in split_rows.items()
            }
            plan["data_addon"] = {
                "name": bundle_path.name, "sha256": digest,
                "added": {split: {label: counters[split, label] for label in FALLBACK_ORDER} for split in SPLITS},
            }
            local_addon = workspace / f"addon-{digest}"
            if not local_addon.is_dir():
                with TemporaryDirectory(dir=workspace) as directory:
                    staging = Path(directory) / "staged"
                    bundle.extractall(staging, sorted(paths))
                    staging.rename(local_addon)
    return plan, local_addon


dataset, addon_images = load_data_addon(prepared, addon_bundle, workspace)
print(f"Loaded {len(dataset['images']):,} photos for {len(dataset['classes'])} labels.")
for split, counts in dataset["fallback_counts"].items():
    print(f"{split} | fallback total {counts} | add-on {dataset['data_addon']['added'][split]}")

## 3. Define EfficientNet-B2 and calculate metrics

In [ ]:
def load_candidate(path, classes):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    image_size = WEIGHTS.transforms().crop_size[0]
    epoch = checkpoint.get("epoch")
    if (checkpoint.get("architecture") != "efficientnet_b2" or checkpoint.get("classes") != classes
            or checkpoint.get("image_size") != image_size or checkpoint.get("smoke") is not False
            or type(epoch) is not int or epoch < 1):
        raise ValueError("Starting checkpoint must be a completed CookLens candidate with the same class order and image size.")
    model = efficientnet_b2(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(classes))
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    return model, epoch


def summarize(confusion, hits, loss, labels):
    actual, predicted = confusion.sum(dim=1), confusion.sum(dim=0)
    correct = confusion.diag()
    precision = correct / predicted.clamp_min(1)
    recall = correct / actual.clamp_min(1)
    f1 = 2 * correct / (actual + predicted).clamp_min(1)

    groups = ("supported_dish", *FALLBACK_ORDER)
    group_index = {name: index for index, name in enumerate(groups)}
    routes = torch.tensor([group_index.get(label, 0) for label in labels])
    route_pairs = routes[:, None] * len(groups) + routes[None, :]
    routing = torch.zeros(len(groups)**2, dtype=torch.long)
    routing.scatter_add_(0, route_pairs.flatten(), confusion.flatten())
    routing = routing.view(len(groups), len(groups))
    supported = routes == 0
    total = int(actual.sum())
    per_class = [
        {"images": int(actual[i]), "predictions": int(predicted[i]), "top1": float(recall[i]),
         "top3": int(hits[i]) / max(1, int(actual[i])), "precision": float(precision[i]),
         "recall": float(recall[i]), "f1": float(f1[i])}
        for i in range(len(labels))
    ]
    return {"images": total, "loss": loss, "top1": int(correct.sum()) / total,
            "top3": int(hits.sum()) / total, "macro_f1": float(f1[actual > 0].mean()),
            "supported_top3": int(hits[supported].sum()) / max(1, int(actual[supported].sum())),
            "per_class": per_class, "confusion": confusion.tolist(),
            "routing_groups": list(groups), "routing_confusion": routing.tolist()}


def print_scores(title, metrics, labels):
    print(f"{title} | loss {metrics['loss']:.4f} | top-1 {metrics['top1']:.2%} | "
          f"top-3 {metrics['top3']:.2%} | macro F1 {metrics['macro_f1']:.4f}", flush=True)
    for label in FALLBACK_ORDER:
        row = metrics["per_class"][labels.index(label)]
        print(f"  {label:<12} precision {row['precision']:.2%} | recall {row['recall']:.2%} | "
              f"F1 {row['f1']:.4f} | {row['images']} photos", flush=True)


def process_epoch(model, loader, loss_fn, device, labels, normalization, *, stage,
                  optimizer=None, blocks=(), scaler=None, capture_scores=False):
    training = optimizer is not None
    gpu_run = device.type == "cuda"
    model.train(training)
    if training:
        model.features.eval()
        for block in blocks:
            block.train()

    loss_total = torch.zeros((), device=device)
    class_count = len(labels)
    if not training:
        confusion = torch.zeros(class_count**2, device=device, dtype=torch.long)
        hits = torch.zeros(class_count, device=device, dtype=torch.long)
    score_batches = []
    expected_photos = len(loader.batch_sampler.indices)
    progress = Progress(stage, expected_photos)
    completed = 0
    if gpu_run:
        torch.cuda.reset_peak_memory_stats(device)
    with torch.set_grad_enabled(training):
        for images, targets in loader:
            images = images.to(device, non_blocking=gpu_run,
                               memory_format=torch.channels_last if gpu_run else torch.preserve_format)
            images = images.float().div_(255)
            mean, std = normalization
            images.sub_(mean).div_(std)
            targets = targets.to(device, non_blocking=gpu_run)
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=training and gpu_run):
                logits = model(images)
                loss = loss_fn(logits, targets)
            loss_total.add_(loss.detach() * len(targets))

            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                choices = logits.topk(3, dim=1).indices
                predictions = choices[:, 0]
                correct = (targets[:, None] == choices).any(dim=1)
                confusion.scatter_add_(0, targets * class_count + predictions, torch.ones_like(targets))
                hits.scatter_add_(0, targets, correct.long())
                if capture_scores:
                    confidence = logits.softmax(dim=1).gather(1, predictions[:, None]).squeeze(1)
                    score_batches.append(torch.stack((targets, predictions, confidence), dim=1))
            completed += len(targets)
            if progress.due(completed):
                detail = f" | loss {(loss_total / completed).item():.4f}" if training else ""
                if gpu_run:
                    peak = torch.cuda.max_memory_allocated(device) / 2**30
                    detail += f" | peak PyTorch VRAM {peak:.2f} GiB"
                progress.update(completed, detail)

    mean_loss = loss_total.item() / completed
    if training:
        return {"loss": mean_loss}
    result = summarize(confusion.view(class_count, class_count).cpu(), hits.cpu(), mean_loss, labels)
    if capture_scores:
        result["scores"] = [[int(actual), int(predicted), score]
                            for actual, predicted, score in torch.cat(score_batches).cpu().tolist()]
    return result

## 4. Train and select the best checkpoint

In [ ]:
def train_candidate(plan, archives, fallback, addon_images, checkpoint, output, *,
                    device, extra_epochs, feature_lr, head_lr, lr_patience, stop_patience,
                    min_f1_gain, batch_size, eval_batch_size, threads, max_workers, seed=67):
    torch.manual_seed(seed)
    torch.set_num_threads(threads)
    labels, baseline = plan["classes"], plan["baseline_classes"]
    gpu_run = device.type == "cuda"
    if labels[:baseline] != BASELINE_CLASSES or labels[baseline:baseline + len(FALLBACK_ORDER)] != list(FALLBACK_ORDER):
        raise ValueError("The audited plan must preserve the original dish and fallback class order.")
    if extra_epochs < 1 or lr_patience < 0 or stop_patience < lr_patience + 2 or min_f1_gain <= 0:
        raise ValueError("Use at least one epoch and leave time to train after the learning rate drops.")
    network, starting_epoch = load_candidate(checkpoint, labels)
    network.to(device)
    last_epoch = starting_epoch + extra_epochs
    print(f"Continuing from epoch {starting_epoch} | checkpoint {checkpoint}", flush=True)
    if gpu_run:
        network.to(memory_format=torch.channels_last)
    scaler = torch.amp.GradScaler("cuda", enabled=gpu_run)
    rows = collect_rows(plan)
    output.mkdir(parents=True, exist_ok=False)
    eval_preset = WEIGHTS.transforms()
    size = eval_preset.crop_size[0]
    eval_transform = transforms.Compose([
        transforms.Resize(eval_preset.resize_size, interpolation=eval_preset.interpolation,
                          antialias=eval_preset.antialias),
        transforms.CenterCrop(eval_preset.crop_size),
        transforms.PILToTensor(),
    ])
    normalization = tuple(torch.tensor(channel_stats, device=device).view(1, 3, 1, 1)
                          for channel_stats in (eval_preset.mean, eval_preset.std))
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(size, scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
        transforms.PILToTensor(),
    ])
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    network.features.requires_grad_(False)
    blocks = tuple(network.features[-2:])
    for block in blocks:
        block.requires_grad_(True)
    optimizer = torch.optim.AdamW([
        {"params": (parameter for block in blocks for parameter in block.parameters()),
         "lr": feature_lr, "name": "features"},
        {"params": network.classifier.parameters(), "lr": head_lr, "name": "head"},
    ], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.3, patience=lr_patience, threshold=min_f1_gain, threshold_mode="abs",
    )
    history = []
    artifact = output / "efficientnet_b2_pilot.pt"
    # Each worker can keep one prefetched batch in shared memory. Reserve space
    # for a batch in transit and leave room for other Colab processes.
    shared_memory = Path("/dev/shm")
    workers = max_workers if shared_memory.is_dir() else 0
    if workers:
        # Keep the host queue uint8. Float conversion and normalization happen on the GPU.
        bytes_per_batch = max(batch_size, eval_batch_size) * (3 * size**2 + 8)
        available = shutil.disk_usage(shared_memory).free // 2
        workers = min(workers, max(0, available // bytes_per_batch - 1))
    settings = {"train_batch_size": batch_size, "eval_batch_size": eval_batch_size,
                "workers": workers, "loader_input": "uint8", "training_precision": "mixed float16" if gpu_run else "float32",
                "extra_epochs": extra_epochs, "feature_lr": feature_lr, "head_lr": head_lr,
                "lr_patience": lr_patience, "stop_patience": stop_patience, "min_f1_gain": min_f1_gain}
    print(f"Run settings | train batch {batch_size} | eval batch {eval_batch_size} | "
          f"{workers} loader {'workers' if workers != 1 else 'worker'} | "
          f"uint8 host batches | {settings['training_precision']}", flush=True)
    loader_options = {"pin_memory": gpu_run, "num_workers": workers}
    if workers:
        loader_options.update(prefetch_factor=1, persistent_workers=True, multiprocessing_context="fork")

    with ExitStack() as resources:
        sets = {}
        for split, items in rows.items():
            pictures = PilotImages(items, archives / f"{split}.zip", fallback, addon_images,
                                   train_transform if split == "train" else eval_transform)
            sets[split] = pictures
            resources.callback(pictures.close)

        combined = ConcatDataset([sets[split] for split in SPLITS])
        offsets = dict(zip(SPLITS, [0, *combined.cumulative_sizes[:-1]]))
        phase_batches = PhaseBatches()
        loader = DataLoader(combined, batch_sampler=phase_batches, **loader_options)

        def run_phase(split, stage, train_indices=None, **training_options):
            offset = offsets[split]
            if train_indices is None:
                indices = range(offset, offset + len(sets[split]))
                phase_size = eval_batch_size
            else:
                indices = [offset + index for index in train_indices]
                phase_size = batch_size
            phase_batches.select(indices, phase_size)
            return process_epoch(network, loader, loss_fn, device, labels, normalization,
                                 stage=stage, **training_options)

        initial = run_phase("validation", f"Epoch {starting_epoch} starting validation", capture_scores=True)
        best_validation = initial.copy()
        best_scores = best_validation.pop("scores")
        best_score = plateau_best = best_validation["macro_f1"]
        best_epoch = starting_epoch
        scheduler.step(best_score)
        print_scores(f"Epoch {starting_epoch} starting validation complete", best_validation, labels)

        temporary = artifact.with_suffix(".tmp")
        shutil.copyfile(checkpoint, temporary)
        temporary.replace(artifact)
        print(f"Saved the starting checkpoint from epoch {starting_epoch}.", flush=True)

        groups = defaultdict(list)
        for i, row in enumerate(rows["train"]):
            groups[row["class_index"]].append(i)
        draws = max(1, len(rows["train"]) // (len(labels) + 1))
        stalled_epochs = 0
        stop_reason = "epoch_limit"
        for epoch in range(starting_epoch + 1, last_epoch + 1):
            rates = {group["name"]: group["lr"] for group in optimizer.param_groups}
            print(f"Epoch {epoch}/{last_epoch} | feature LR {rates['features']:.2g} | head LR {rates['head']:.2g}",
                  flush=True)
            indices = sample_epoch(groups, baseline + 1, draws, seed + epoch)
            loss = run_phase("train", f"Epoch {epoch}/{last_epoch} training", train_indices=indices,
                             optimizer=optimizer, blocks=blocks, scaler=scaler)["loss"]
            metrics = run_phase("validation", f"Epoch {epoch}/{last_epoch} validation", capture_scores=True)
            scores = metrics.pop("scores")
            score = metrics["macro_f1"]
            history.append({"epoch": epoch, "train_loss": loss, "learning_rates": rates, "validation": metrics})
            print_scores(f"Epoch {epoch}/{last_epoch} validation complete | train loss {loss:.4f}", metrics, labels)

            if score > best_score:
                best_score, best_epoch, best_validation, best_scores = score, epoch, metrics, scores
                torch.save({
                    "architecture": "efficientnet_b2", "image_size": size, "classes": labels,
                    "model_state_dict": network.state_dict(), "epoch": epoch, "smoke": False,
                }, temporary)
                temporary.replace(artifact)
                print(f"Saved best checkpoint from epoch {epoch}.", flush=True)

            scheduler.step(score)
            next_rates = {group["name"]: group["lr"] for group in optimizer.param_groups}
            if next_rates != rates:
                print(f"Reduced learning rates | features {next_rates['features']:.2g} | head {next_rates['head']:.2g}",
                      flush=True)
            if score > plateau_best + min_f1_gain:
                plateau_best = score
                stalled_epochs = 0
            else:
                stalled_epochs += 1
                if stalled_epochs >= stop_patience:
                    stop_reason = "validation_plateau"
                    print(f"Early stop after {stalled_epochs} epochs without a meaningful macro F1 gain.", flush=True)
                    break

        selected, selected_epoch = load_candidate(artifact, labels)
        if selected_epoch != best_epoch:
            raise RuntimeError("Saved checkpoint epoch differs from validation selection.")
        network.load_state_dict(selected.state_dict(), strict=True)
        test = run_phase("test", f"Held-out test (epoch {best_epoch})", capture_scores=True)
        fresh_test = {label: {"photos": 0, "correct": 0} for label in FALLBACK_ORDER}
        for row, (actual, predicted, _) in zip(rows["test"], test.pop("scores"), strict=True):
            if row.get("origin") in ("round2_other_food", "round2_non_food"):
                group = fresh_test[labels[actual]]
                group["photos"] += 1
                group["correct"] += int(actual == predicted)
        del loader

    report = {"classes": labels, "device": str(device), "smoke": False, "settings": settings,
              "starting_checkpoint": str(checkpoint), "starting_epoch": starting_epoch,
              "history": history, "best_epoch": best_epoch, "best_validation": best_validation,
              "validation_scores": {"columns": ["actual_class", "predicted_class", "top1_probability"],
                                    "rows": best_scores},
              "stop_reason": stop_reason, "data_addon": plan["data_addon"], "test": test, "fresh_test": fresh_test}
    (output / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


train_candidate(dataset, archives, fallback, addon_images, checkpoint, output, device=torch.device("cuda"),
                extra_epochs=extra_epochs, feature_lr=feature_lr, head_lr=head_lr, lr_patience=lr_patience,
                stop_patience=stop_patience, min_f1_gain=min_f1_gain, batch_size=batch_size,
                eval_batch_size=eval_batch_size, threads=threads, max_workers=max_workers)

## 5. Read the held-out evaluation

In [ ]:
report_file = output / "report.json"
weights_file = output / "efficientnet_b2_pilot.pt"
report = json.loads(report_file.read_text(encoding="utf-8"))
if report["smoke"] or not weights_file.is_file():
    raise RuntimeError("The full training run did not produce its expected artifacts.")
if report["device"] != "cuda":
    raise RuntimeError(f"Training did not use the selected Colab GPU: {report['device']}.")

test = report["test"]
print(f"{device_name} | selected epoch {report['best_epoch']} | started at epoch {report['starting_epoch']}")
print_scores(f"Held-out test on {test['images']:,} photos", test, report["classes"])
print(f"Supported dishes top-3 {test['supported_top3']:.2%}")
print()
print(f"{'Actual / predicted':<21}" + "".join(f"{name:>16}" for name in test["routing_groups"]))
for name, row in zip(test["routing_groups"], test["routing_confusion"]):
    print(f"{name:<21}" + "".join(f"{value:>16,}" for value in row))
print()
for label, counts in report["fresh_test"].items():
    print(f"Fresh {label} test photos correctly identified | {counts['correct']}/{counts['photos']}")
print("Checkpoint", weights_file)
print("Report with validation history, scores, and per-class metrics", report_file)